In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:

# load_dotenv will define all variables from the .env files as environment variables
load_dotenv(override=True)

# Here we read the API key from the environment variable
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key defined in the .env file! Please add your OpenAI API key to the .env file - see troubleshooting notebook")
elif not api_key.startswith("sk-proj-"):
    print("The API key has the wrong format - it should start with 'sk-proj-'")
else:
    print("API key was defined with the correct format!")

In [ ]:
from jobspy import scrape_jobs
import pandas as pd

# 1. Scrape with quotes to encourage exact phrase matching
jobs = scrape_jobs(
    site_name=["linkedin", "indeed", "glassdoor", "google"],
    search_term='"AI Engineer"', # Note the single quotes around double quotes
    location="USA",
    results_wanted=30, 
)

# 2. Convert to DataFrame
df = pd.DataFrame(jobs)

# 3. APPLY THE STRICT FILTER
# We ensure the title contains BOTH "AI" and "Engineer" (case-insensitive)
mask = df['title'].str.contains('AI', case=False, na=False) & \
       df['title'].str.contains('Engineer', case=False, na=False)

filtered_jobs = df[mask].copy()

# 4. Result
print(f"Total jobs found: {len(df)}")
print(f"Jobs matching 'AI' + 'Engineer' in title: {len(filtered_jobs)}")

display(filtered_jobs[['title', 'company']].head())

In [ ]:
import re

if filtered_jobs.empty:
    raise ValueError("filtered_jobs is empty. Please adjust your filters before summarizing.")

if "description" not in filtered_jobs.columns:
    raise KeyError("filtered_jobs must contain a 'description' column.")

summary_source_jobs = filtered_jobs.copy()
summary_source_jobs = summary_source_jobs[summary_source_jobs["description"].notna()].copy()
summary_source_jobs["description"] = summary_source_jobs["description"].astype(str).str.strip()
summary_source_jobs = summary_source_jobs[summary_source_jobs["description"] != ""].copy()

if "job_url" in summary_source_jobs.columns:
    summary_source_jobs = summary_source_jobs.drop_duplicates(subset=["job_url"])
else:
    dedupe_columns = [
        column
        for column in ["title", "company", "description"]
        if column in summary_source_jobs.columns
    ]
    summary_source_jobs = summary_source_jobs.drop_duplicates(subset=dedupe_columns)

if summary_source_jobs.empty:
    raise ValueError("filtered_jobs contains no usable job descriptions after cleaning.")

def clean_description(text):
    text = str(text)
    text = text.replace("**", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

job_entries = []

for _, job in summary_source_jobs.iterrows():
    description = clean_description(job["description"])
    if not description:
        continue

    title = str(job.get("title", "Unknown title")).strip()
    company = str(job.get("company", "Unknown company")).strip()
    job_url = str(job.get("job_url", "")).strip()

    entry_lines = [
        f"Job {len(job_entries) + 1}",
        f"Title: {title}",
        f"Company: {company}",
        f"Description: {description}",
    ]

    if job_url and job_url.lower() != "nan":
        entry_lines.append(f"URL: {job_url}")

    job_entries.append("\n".join(entry_lines))

if not job_entries:
    raise ValueError("No non-empty job descriptions were available for summarization.")

job_descriptions = "\n\n---\n\n".join(job_entries)

prompt = f"""
You are analyzing scraped job postings for AI Engineer roles.

Read every job description below and summarize the shared responsibilities across the roles.
Ignore company marketing, benefits, equal opportunity text, visa/location boilerplate, and generic recruiting language.
Focus on repeated responsibilities, expected day-to-day work, collaboration patterns, and technical ownership.

Return your answer in this format:
## Core responsibilities
- 8-12 concise bullet points

## Role focus
- One short paragraph

## Notable differences
- 3-5 bullet points for responsibilities that only appear in some postings

## Machine Learning Knowledge Required
- Describe how much knowledge about Machine Leaning (building models from scratch vs. using pre-trained models, etc.) is expected across the job postings

## Use Cases
- Describe the most common use cases mentioned across the job postings

Job descriptions:
{job_descriptions}
""".strip()

messages = [
    {
        "role": "system",
        "content": "You analyze scraped job postings and identify the true recurring responsibilities across them.",
    },
    {"role": "user", "content": prompt},
]

print(f"Summarizing {len(job_entries)} scraped job descriptions")
print(f"Combined description length: {len(job_descriptions):,} characters")
messages

In [ ]:
client = OpenAI()

response = client.chat.completions.create(model="gpt-5-mini", messages=messages)
summary = response.choices[0].message.content
print(summary)